# Assignment 2

In [1]:
# setup
import pandas as pd
from scipy import stats
from scipy.stats.mstats import mquantiles

### Applying our changes from Assignment 1:

In [2]:
# data
transaction = pd.read_csv("../data/transaction_data.csv")
transaction

#dataframe format
transaction.columns = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]   #from 2_data_cleaning

#transaction data type fix
transaction["transaction_date"] = pd.to_datetime(transaction["transaction_date"], format='%Y-%m-%d') # from 2_data_cleaning
transaction.info()

# fix lead time
transaction[transaction['lead_time'] == 2800]
transaction.loc[
    transaction['lead_time'] == 2800,
    'lead_time'] = 28

#unit price and order quantity values fix
transaction['unit_price'] = transaction['unit_price'].abs()
transaction['order_quantity'] = transaction['order_quantity'].abs()

#Na values
Transaction_ValuesMissing = ['sku_number','inventory_type', 'stocking_type', 'manufacturing_site', 'division_code'] 
transaction[Transaction_ValuesMissing] = transaction[Transaction_ValuesMissing].fillna("missing")
transaction


<class 'pandas.DataFrame'>
RangeIndex: 784639 entries, 0 to 784638
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   sku_number          784469 non-null  str           
 1   inventory_type      775250 non-null  str           
 2   stocking_type       763084 non-null  str           
 3   lead_time           784639 non-null  int64         
 4   unit_price          784639 non-null  float64       
 5   manufacturing_site  632214 non-null  str           
 6   division_code       729751 non-null  str           
 7   transaction_date    784639 non-null  datetime64[us]
 8   order_quantity      784639 non-null  int64         
dtypes: datetime64[us](1), float64(1), int64(2), str(5)
memory usage: 53.9 MB


,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
0,DB45C19C,FG,MTO,28,992.860458,JP-41,missing,2023-01-01,130
1,83992E02,missing,MTS,21,202.017834,CA-66,ACA9,2023-01-01,39
2,75C364B0,FG,MTS,28,986.009314,JP-41,23F0,2023-01-01,111
3,2C9772C9,missing,MTS,28,1007.216590,CA-66,9B75,2023-01-01,89
4,ACCCE163,WIP,MTS,28,986.998503,CA-66,6E75,2023-01-01,80
...,...,...,...,...,...,...,...,...,...
784634,F8B91C1B,WIP,MTS,28,998.021118,missing,23F0,2025-12-30,111
784635,6664F459,FG,MTS,28,99.200335,CA-66,D66F,2025-12-30,1
784636,3DE1D314,FG,MTS,28,1008.523300,missing,81FE,2025-12-30,115
784637,DB131462,FG,MTS,28,1008.340512,CA-2B,6AF1,2025-12-30,122


The changes that were mentioned in Assignment 1 have been made through this shell above. We now don't have any data loss as we have changed NA values into "missing". We are now ready to start the safety stock calculation process.

## 1. Required Transformation

In [3]:
mts = transaction.query('stocking_type in "MTS"')
mts

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
1,83992E02,missing,MTS,21,202.017834,CA-66,ACA9,2023-01-01,39
2,75C364B0,FG,MTS,28,986.009314,JP-41,23F0,2023-01-01,111
3,2C9772C9,missing,MTS,28,1007.216590,CA-66,9B75,2023-01-01,89
4,ACCCE163,WIP,MTS,28,986.998503,CA-66,6E75,2023-01-01,80
14,D89A6000,FG,MTS,28,988.743492,US-B4,219B,2023-01-01,75
...,...,...,...,...,...,...,...,...,...
784634,F8B91C1B,WIP,MTS,28,998.021118,missing,23F0,2025-12-30,111
784635,6664F459,FG,MTS,28,99.200335,CA-66,D66F,2025-12-30,1
784636,3DE1D314,FG,MTS,28,1008.523300,missing,81FE,2025-12-30,115
784637,DB131462,FG,MTS,28,1008.340512,CA-2B,6AF1,2025-12-30,122


Items with the stocking type `MTS` were filtered and stored in a separate dataframe.

In [4]:
#Create new data frame grouped by sku and aggregated quantity and lead time for SS calculation
mts_df =(mts
         .groupby("sku_number")
         .agg(min_qty=("order_quantity", "min"),
              max_qty=("order_quantity", "max"),
              avg_qty=("order_quantity", "mean"),
              median_qty=("order_quantity", "median"),
              variance_qty=("order_quantity", "var"),
              std_qty=("order_quantity", "std"),
              avg_lead_time=("lead_time", "mean"))
              .reset_index())
mts_df

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time
0,00234778,0,36,9.117647,7.0,58.679091,7.660228,14.000000
1,01517B90,24,169,100.552604,100.0,420.922537,20.516397,28.000000
2,020287B4,0,77,24.360577,22.0,273.990153,16.552648,21.000000
3,02144443,13,167,99.677966,101.0,409.478645,20.235579,28.000000
4,0273F5A5,0,86,23.654639,21.5,281.229255,16.769891,21.000000
...,...,...,...,...,...,...,...,...
504,FD816E1D,0,76,20.574737,17.0,254.785014,15.961987,19.526316
505,FE482594,0,83,23.474801,21.0,282.936198,16.820707,21.000000
506,FE564816,0,38,10.760417,9.0,68.172666,8.256674,28.000000
507,FEAD0FF0,0,169,87.996196,95.0,1234.273059,35.132223,26.315739


**Explanation:**

Creating this table provides us the calculations we need to prepare for our safety stock calculations. This table includes insightful statistic features to see the minimum, maximum, average, median, and variance quantities for each SKU. 

## 2. Safety Stock Calculation

In [5]:
#Set the z-score for each service level, under a normal distribution
mts_z75 = stats.norm.ppf(.75)
mts_z90 = stats.norm.ppf(.90)
mts_z95 = stats.norm.ppf(.95)

print(mts_z75, mts_z90, mts_z95)

0.6744897501960817 1.2815515655446004 1.644853626951472


This gives us ths the z-scores for each service level. It is 0.674 for 75%, 1.282 for 90%, and 1.645 for 95%. We will use these values for the saftey stock calculation. These will always be the same, no matter what data we are working with. 

In [6]:
#Add column for safety stock at 75% service level
mts_df["safety_stock_75"] = (
    mts_z75
    *mts_df["std_qty"]
    *mts_df["avg_lead_time"]** 0.5      #The "** 0.5" calculates the square root by applying the power of 0.5 to the value. (same as square root)
)
mts_df

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time,safety_stock_75
0,00234778,0,36,9.117647,7.0,58.679091,7.660228,14.000000,19.332190
1,01517B90,24,169,100.552604,100.0,420.922537,20.516397,28.000000,73.224339
2,020287B4,0,77,24.360577,22.0,273.990153,16.552648,21.000000,51.162585
3,02144443,13,167,99.677966,101.0,409.478645,20.235579,28.000000,72.222081
4,0273F5A5,0,86,23.654639,21.5,281.229255,16.769891,21.000000,51.834063
...,...,...,...,...,...,...,...,...,...
504,FD816E1D,0,76,20.574737,17.0,254.785014,15.961987,19.526316,47.574305
505,FE482594,0,83,23.474801,21.0,282.936198,16.820707,21.000000,51.991130
506,FE564816,0,38,10.760417,9.0,68.172666,8.256674,28.000000,29.468600
507,FEAD0FF0,0,169,87.996196,95.0,1234.273059,35.132223,26.315739,121.559462


In [7]:
#Add column for safety stock at 90% service level
mts_df["safety_stock_90"] = (
    mts_z90
    *mts_df["std_qty"]
    *mts_df["avg_lead_time"]** 0.5 
)
mts_df

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time,safety_stock_75,safety_stock_90
0,00234778,0,36,9.117647,7.0,58.679091,7.660228,14.000000,19.332190,36.731765
1,01517B90,24,169,100.552604,100.0,420.922537,20.516397,28.000000,73.224339,139.128528
2,020287B4,0,77,24.360577,22.0,273.990153,16.552648,21.000000,51.162585,97.210507
3,02144443,13,167,99.677966,101.0,409.478645,20.235579,28.000000,72.222081,137.224207
4,0273F5A5,0,86,23.654639,21.5,281.229255,16.769891,21.000000,51.834063,98.486336
...,...,...,...,...,...,...,...,...,...,...
504,FD816E1D,0,76,20.574737,17.0,254.785014,15.961987,19.526316,47.574305,90.392663
505,FE482594,0,83,23.474801,21.0,282.936198,16.820707,21.000000,51.991130,98.784769
506,FE564816,0,38,10.760417,9.0,68.172666,8.256674,28.000000,29.468600,55.991260
507,FEAD0FF0,0,169,87.996196,95.0,1234.273059,35.132223,26.315739,121.559462,230.966769


In [8]:
#Add column for safety stock at 95% service level
mts_df["safety_stock_95"] = (
    mts_z95
    *mts_df["std_qty"]
    *mts_df["avg_lead_time"]** 0.5 
)
mts_df

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
0,00234778,0,36,9.117647,7.0,58.679091,7.660228,14.000000,19.332190,36.731765,47.144710
1,01517B90,24,169,100.552604,100.0,420.922537,20.516397,28.000000,73.224339,139.128528,178.569533
2,020287B4,0,77,24.360577,22.0,273.990153,16.552648,21.000000,51.162585,97.210507,124.768336
3,02144443,13,167,99.677966,101.0,409.478645,20.235579,28.000000,72.222081,137.224207,176.125363
4,0273F5A5,0,86,23.654639,21.5,281.229255,16.769891,21.000000,51.834063,98.486336,126.405844
...,...,...,...,...,...,...,...,...,...,...,...
504,FD816E1D,0,76,20.574737,17.0,254.785014,15.961987,19.526316,47.574305,90.392663,116.017727
505,FE482594,0,83,23.474801,21.0,282.936198,16.820707,21.000000,51.991130,98.784769,126.788879
506,FE564816,0,38,10.760417,9.0,68.172666,8.256674,28.000000,29.468600,55.991260,71.864004
507,FEAD0FF0,0,169,87.996196,95.0,1234.273059,35.132223,26.315739,121.559462,230.966769,296.442639


**Explanation:**

The code above has added the safety stock under three different confidence levels consisting of 75%, 90%, and 95%. This allows us to see different levels of safety stock we would need to keep into consideration for the make to stock (`MTS`) SKU items. 

## 3. Safety Stock Distribution

In [9]:
#95% safety stock
#Sort for largest
mts_df.sort_values(by="safety_stock_95", ascending=False).head(1)

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
484,F39BBC11,0,161,77.029494,91.0,1900.860374,43.598857,26.487844,151.346982,287.563988,369.084382


At a 95% service level SKU `F39BBC11` has the largest safety stock, with a quantity of 369.084382, which we will round to 370 units. 

In [10]:
#Sort for smallest
mts_df.sort_values(by="safety_stock_95").head(1)

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
90,2462CBBC,0,22,7.840426,7.0,24.84523,4.984499,14.0,12.579428,23.901305,30.676993


At a 95% service level SKU `2462CBBC` has the smallest safety stock, with a quantity of 30.676993, which we will round to 31 units. 

In [11]:
#Mean of safety stock column across all SKUs
mts_df["safety_stock_95"].mean()

np.float64(124.1958689900739)

The average safety stock across all SKU's is 124.2 units at a 95% service level. 

## Side Quest

We grouped `MTS` items by `SKU` and `transaction_date` and found total order quantities, and set it to `daily_order_qty`.

In [12]:
daily_order_qty = (
    mts.groupby(["sku_number", "transaction_date"], as_index=False)
    ["order_quantity"].sum()
)
daily_order_qty

,sku_number,transaction_date,order_quantity
0,00234778,2023-01-21,31
1,00234778,2023-01-28,7
2,00234778,2023-02-23,7
3,00234778,2023-02-24,5
4,00234778,2023-03-01,3
...,...,...,...
227198,FF6FAC01,2025-12-25,267
227199,FF6FAC01,2025-12-26,182
227200,FF6FAC01,2025-12-27,85
227201,FF6FAC01,2025-12-29,239


 Then we used `mquantiles()` from SciPy to find the 0.75, 0.90, and 0.95 percentiles for each SKU. We learned that empirical quantiles actually use the historical demand instead of just assuming it follows the normal distribution bell shaped curve.  

In [13]:
quantiles = pd.DataFrame(columns = ["sku_number", "safety_stock_75", "safety_stock_90", "safety_stock_95"])
orders = daily_order_qty[["sku_number", "order_quantity"]]
for sku in orders["sku_number"].unique():
	prob = mquantiles(orders.query('sku_number == @sku')["order_quantity"], prob = [0.75, 0.9, 0.95], alphap = 0, betap = 1)
	quantiles.loc[len(quantiles)] = [sku, prob[0], prob[1], prob[2]]
quantiles.sort_values("safety_stock_95", ascending = False)

,sku_number,safety_stock_75,safety_stock_90,safety_stock_95
5,030D6C59,656.00,824.0,927.00
252,7B294AD6,651.00,803.7,909.80
184,58C39130,467.00,597.7,693.35
283,91F6F23F,455.75,604.6,692.90
494,F8B91C1B,471.00,618.0,692.40
...,...,...,...,...
141,43C96AB6,14.00,18.0,18.50
127,3A5E9C6B,11.00,17.0,18.00
175,53F0A631,11.25,15.3,17.65
211,61D81899,12.50,16.0,17.50


# References:
- Bobbitt, Z. (2021, April 26). Percentile vs. quartile vs. quantile: What’s the difference? Statology. https://www.statology.org/percentile-vs-quartile-vs-quantile/

We used this to understand quantiles and how we can relate it to safety stock calculation. 

- Stack Overflow(2022). "How do I calculate square root in Python?". Retrieved November 17, 2026. https://stackoverflow.com/questions/70793490/how-do-i-calculate-square-root-in-python 

When searching for "How to square root in python" on Stack Overflow, we come across a post (Stack Overflow, 2022) that has last been edited this year on August 24th that indicates the snippet below:

```
Option 2: Fractional exponent
The power operator (**) or the built-in pow() function can also be used to calculate a square root. Mathematically speaking, the square root of a equals a to the power of 1/2.

The power operator requires numeric types and matches the conversion rules for binary arithmetic operators, so in this case it will return either a float or a complex number

>>> 9 ** (1/2)
3.0
>>> 9 ** .5  # Same thing
3.0
>>> 2 ** .5
1.4142135623730951

```